In [1]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [2]:
sql = '''
SELECT "DISTRICT",
       "OCCURRED_ON_DATE",
       "SHOOTING",
       "OFFENSE_DESCRIPTION"
FROM "b973d8cb-eeb2-4e7e-99da-c92938efc9c0"
WHERE CAST("SHOOTING" AS INT)=1
'''

In [3]:
bpd_logs = "https://data.boston.gov/api/3/action/datastore_search_sql"

In [4]:
import urllib, requests

In [5]:
url = bpd_logs + "?sql=" + urllib.parse.quote(sql)
rows = requests.get(url).json()

shootings = rows['result']['records']

In [6]:
shootings[:5]

[{'DISTRICT': 'B3',
  'OCCURRED_ON_DATE': '2023-02-18 21:43:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'MURDER, NON-NEGLIGENT MANSLAUGHTER'},
 {'DISTRICT': 'D4',
  'OCCURRED_ON_DATE': '2023-05-24 18:49:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'BALLISTICS EVIDENCE/FOUND'},
 {'DISTRICT': 'D4',
  'OCCURRED_ON_DATE': '2023-10-27 12:50:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'ASSAULT - AGGRAVATED'},
 {'DISTRICT': 'B3',
  'OCCURRED_ON_DATE': '2023-01-07 17:51:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'INVESTIGATE PROPERTY'},
 {'DISTRICT': 'B2',
  'OCCURRED_ON_DATE': '2023-01-30 17:00:00+00',
  'SHOOTING': '1',
  'OFFENSE_DESCRIPTION': 'ASSAULT - AGGRAVATED'}]

In [7]:
from datetime import datetime

clean_data = []

for s in shootings:
    if s["OCCURRED_ON_DATE"] and s["DISTRICT"]:
        year = datetime.strptime(s["OCCURRED_ON_DATE"], "%Y-%m-%d %H:%M:%S+00").year

        clean_data.append({
            "district": s["DISTRICT"],
            "year": year,
            "offense": s["OFFENSE_DESCRIPTION"]
        })

In [8]:
from collections import defaultdict

dataset = defaultdict(lambda: defaultdict(int))

for item in clean_data:
    district = item["district"]
    year = item["year"]

    dataset[district][year] += 1

In [10]:
offense_data = []

for item in clean_data:
    offense_data.append({
        "district": item["district"],
        "offense": item["offense"]
    })

In [11]:
final_dataset = {
    "district_year": dataset,
    "offenses": offense_data
}

In [12]:
final_dataset

{'district_year': defaultdict(<function __main__.<lambda>()>,
             {'B3': defaultdict(int,
                          {2023: 150, 2024: 104, 2025: 147, 2026: 20}),
              'D4': defaultdict(int, {2023: 34, 2024: 19, 2025: 30, 2026: 3}),
              'B2': defaultdict(int,
                          {2023: 150, 2025: 135, 2024: 109, 2026: 23}),
              'C11': defaultdict(int,
                          {2023: 131, 2024: 103, 2025: 87, 2026: 15}),
              'D14': defaultdict(int, {2025: 18, 2023: 11, 2024: 11, 2026: 1}),
              'E18': defaultdict(int, {2024: 34, 2023: 41, 2025: 34, 2026: 1}),
              'A1': defaultdict(int, {2024: 11, 2023: 13, 2025: 14, 2026: 2}),
              'E13': defaultdict(int, {2024: 36, 2023: 55, 2025: 24, 2026: 4}),
              'A7': defaultdict(int, {2024: 8, 2023: 13, 2025: 7, 2026: 1}),
              'E5': defaultdict(int, {2024: 20, 2023: 13, 2025: 18, 2026: 6}),
              'C6': defaultdict(int, {2023: 22, 2024: 20,

In [13]:
def convert_to_dict(d):
    if isinstance(d, defaultdict):
        d = {k: convert_to_dict(v) for k, v in d.items()}
    return d

clean_dataset = convert_to_dict(dataset)

In [14]:
final_dataset = {
    "district_year": clean_dataset,
    "offenses": offense_data
}

In [15]:
final_dataset

{'district_year': {'B3': {2023: 150, 2024: 104, 2025: 147, 2026: 20},
  'D4': {2023: 34, 2024: 19, 2025: 30, 2026: 3},
  'B2': {2023: 150, 2025: 135, 2024: 109, 2026: 23},
  'C11': {2023: 131, 2024: 103, 2025: 87, 2026: 15},
  'D14': {2025: 18, 2023: 11, 2024: 11, 2026: 1},
  'E18': {2024: 34, 2023: 41, 2025: 34, 2026: 1},
  'A1': {2024: 11, 2023: 13, 2025: 14, 2026: 2},
  'E13': {2024: 36, 2023: 55, 2025: 24, 2026: 4},
  'A7': {2024: 8, 2023: 13, 2025: 7, 2026: 1},
  'E5': {2024: 20, 2023: 13, 2025: 18, 2026: 6},
  'C6': {2023: 22, 2024: 20, 2025: 15, 2026: 4},
  'A15': {2025: 4, 2023: 2, 2024: 9, 2026: 1},
  'External': {2023: 1, 2025: 2}},
 'offenses': [{'district': 'B3',
   'offense': 'MURDER, NON-NEGLIGENT MANSLAUGHTER'},
  {'district': 'D4', 'offense': 'BALLISTICS EVIDENCE/FOUND'},
  {'district': 'D4', 'offense': 'ASSAULT - AGGRAVATED'},
  {'district': 'B3', 'offense': 'INVESTIGATE PROPERTY'},
  {'district': 'B2', 'offense': 'ASSAULT - AGGRAVATED'},
  {'district': 'C11', 'offense

In [17]:
import json

with open("boston_crime_data.json", "w") as f:
    json.dump(final_dataset, f)

In [18]:
from google.colab import files
files.download("boston_crime_data.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>